# FAISS Semantic Search Exploration

This notebook explores the FAISS-based semantic search service for music embeddings.

## Goals
1. Build FAISS index from reduced embeddings
2. Test semantic search queries
3. Explore search results and similarity
4. Compare search performance


In [46]:
# Install FAISS (CPU version)
# Note: Use 'faiss-cpu' not 'faiss' - the old 'faiss' package is deprecated
!pip install faiss-cpu

# For GPU support (if you have CUDA), use: !pip install faiss-gpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [47]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.decomposition import PCA

from crate_analysis import Database
from crate_analysis.faiss_search import FAISSSearchService

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 200)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)


## 1. Initialize Search Service

Set up paths and initialize the FAISS search service.


### Create PCA Transformer (if needed)

If you have the original 768d embeddings, you can create the PCA transformer here:


In [48]:
# UNCOMMENT AND RUN THIS IF YOU HAVE THE ORIGINAL 768D EMBEDDINGS
# 
# original_embeddings_path = data_dir / "embeddings_768d.npy"  # Update path if different
# 
# if original_embeddings_path.exists():
#     print("Loading original 768d embeddings...")
#     embeddings_768d = np.load(original_embeddings_path)
#     print(f"✓ Loaded: {embeddings_768d.shape}")
#     
#     print("\nFitting PCA (this may take a few minutes)...")
#     pca = PCA(n_components=256, random_state=42)
#     embeddings_256d = pca.fit_transform(embeddings_768d)
#     
#     variance_explained = pca.explained_variance_ratio_.sum()
#     print(f"✓ PCA fitted")
#     print(f"  Variance explained: {variance_explained:.3%}")
#     
#     print(f"\nSaving PCA transformer to {pca_path}...")
#     with open(pca_path, 'wb') as f:
#         pickle.dump(pca, f)
#     print(f"✓ Saved!")
# else:
#     print(f"Original embeddings not found at {original_embeddings_path}")
#     print("Please update the path or provide the original 768d embeddings.")

print("Skipping PCA creation (uncomment above if you have original embeddings)")


Skipping PCA creation (uncomment above if you have original embeddings)


## 2. Initialize Search Service

Set up paths and initialize the FAISS search service.


In [49]:
!pip install tqdm 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [50]:
# Set up paths
data_dir = Path.cwd().parent.parent / "data"
embeddings_path = data_dir / "embeddings_256d.npy"
metadata_path = data_dir / "metadata.json"
pca_path = data_dir / "pca_256.pkl"
index_path = data_dir / "embeddings_256d.index"

print(f"Data directory: {data_dir}")
print(f"Embeddings: {embeddings_path.exists()}")
print(f"Metadata: {metadata_path.exists()}")
print(f"PCA model: {pca_path.exists()}")
print(f"Index: {index_path.exists()}")


Data directory: /Users/pooks/Dev/crate/data
Embeddings: True
Metadata: True
PCA model: False
Index: True


## 2.5. Verify Pipeline & Alignment

**CRITICAL**: Let's verify the search pipeline is correct and embeddings are aligned with plays.


In [51]:
# Verify pipeline correctness
print("=" * 80)
print("VERIFYING SEARCH PIPELINE")
print("=" * 80)

# 1. Check embedding normalization
print("\n1. Checking stored embeddings...")
sample_embeddings = search_service.embeddings[:1000]
norms = np.linalg.norm(sample_embeddings, axis=1)
print(f"   Embedding norms (sample of 1000):")
print(f"      Mean: {norms.mean():.6f}")
print(f"      Std: {norms.std():.6f}")
print(f"      Range: [{norms.min():.6f}, {norms.max():.6f}]")
if np.allclose(norms, 1.0, atol=0.01):
    print("   ✓ Embeddings are normalized (norms ≈ 1.0)")
else:
    print("   ⚠️  Embeddings are NOT normalized")

# 2. Check PCA transformer
print("\n2. Checking PCA transformer...")
if search_service.pca is not None:
    print(f"   Components: {search_service.pca.n_components_}")
    print(f"   Explained variance: {search_service.pca.explained_variance_ratio_.sum():.3%}")
    if hasattr(search_service.pca, 'mean_'):
        pca_mean_norm = np.linalg.norm(search_service.pca.mean_)
        print(f"   PCA mean norm: {pca_mean_norm:.6f}")
        if pca_mean_norm < 0.1:
            print("   ⚠️  PCA mean is very small - suggests PCA was fit on normalized data")
        else:
            print("   ✓ PCA mean is significant - suggests PCA was fit on unnormalized data")
else:
    print("   ⚠️  PCA transformer not loaded")

# 3. Test query encoding
print("\n3. Testing query encoding...")
test_query = "psychedelic rock"

# CORRECT method: Normalize BEFORE PCA (since PCA was fit on normalized embeddings)
query_768d_norm = search_service.model.encode([test_query], normalize_embeddings=True)[0]
query_256d = search_service.pca.transform([query_768d_norm])[0]
query_256d = query_256d / np.linalg.norm(query_256d)

print(f"   Query encoding pipeline:")
print(f"      768d (normalized): norm = {np.linalg.norm(query_768d_norm):.6f}")
print(f"      256d (after PCA): norm = {np.linalg.norm(query_256d):.6f}")
print(f"   ✓ Using normalize_embeddings=True before PCA (matches training)")

# 4. Check alignment
print("\n4. Checking alignment...")
print(f"   Embeddings: {len(search_service.embeddings):,}")
print(f"   Plays: {len(plays_df):,}")
if len(search_service.embeddings) == len(plays_df):
    print("   ✓ Counts match - embeddings and plays are aligned")
else:
    print("   ⚠️  MISMATCH - embeddings and plays are NOT aligned!")
    print("      This will cause incorrect results!")

# 5. Test alignment with a known play
print("\n5. Testing alignment with known play...")
# Pick a play and check if its embedding matches
test_play_idx = 0
test_play = plays_df.iloc[test_play_idx]
print(f"   Test play at index {test_play_idx}:")
print(f"      ID: {test_play['id']}")
print(f"      Artist: {test_play['artist']}")
print(f"      Song: {test_play['song']}")

# Get the embedding for this play
play_embedding = search_service.embeddings[test_play_idx]
print(f"   Embedding shape: {play_embedding.shape}")
print(f"   Embedding norm: {np.linalg.norm(play_embedding):.6f}")

# Search for similar plays using this embedding
query_vector = play_embedding.reshape(1, -1)
distances, indices = search_service.index.search(query_vector, 6)  # Get 6 to exclude self

print(f"\n   Most similar plays (excluding self):")
for i, (idx, dist) in enumerate(zip(indices[0][1:6], distances[0][1:6]), 1):  # Skip first (self)
    similar_play = plays_df.iloc[idx]
    print(f"      {i}. [{dist:.4f}] {similar_play['artist']} - {similar_play['song']}")

# 6. Test a simple search to see similarity scores
print("\n6. Testing search with sample query...")
try:
    indices, distances = search_service.search(test_query, k=10)
    print(f"   Top 10 results for '{test_query}':")
    for i, (idx, dist) in enumerate(zip(indices[:10], distances[:10]), 1):
        play = plays_df.iloc[idx]
        print(f"      {i:2d}. [{dist:.4f}] {play['artist']} - {play['song']}")
        if pd.notna(play.get('comment')) and play['comment']:
            comment = str(play['comment'])[:60]
            print(f"          💬 {comment}...")
    print(f"\n   Similarity range: [{distances.min():.4f}, {distances.max():.4f}]")
    if distances.max() < 0.3:
        print("   ⚠️  WARNING: Very low similarity scores - results are likely wrong!")
        print("      Possible issues:")
        print("      - Embeddings and plays are misaligned")
        print("      - Query encoding pipeline is incorrect")
        print("      - PCA transformer doesn't match embeddings")
    elif distances.max() < 0.5:
        print("   ⚠️  Low similarity scores - check encoding pipeline")
    else:
        print("   ✓ Similarity scores look reasonable")
except Exception as e:
    print(f"   ✗ Error: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)


VERIFYING SEARCH PIPELINE

1. Checking stored embeddings...
   Embedding norms (sample of 1000):
      Mean: 0.780785
      Std: 0.052578
      Range: [0.647417, 0.928772]
   ⚠️  Embeddings are NOT normalized

2. Checking PCA transformer...
   Components: 256
   Explained variance: 96.434%
   PCA mean norm: 0.642256
   ✓ PCA mean is significant - suggests PCA was fit on unnormalized data

3. Testing query encoding...
   Query encoding pipeline:
      768d (normalized): norm = 1.000000
      256d (after PCA): norm = 1.000000
   ✓ Using normalize_embeddings=True before PCA (matches training)

4. Checking alignment...
   Embeddings: 2,193,235
   Plays: 2,193,235
   ✓ Counts match - embeddings and plays are aligned

5. Testing alignment with known play...
   Test play at index 0:
      ID: 1
      Artist: Peter Gabriel
      Song: Games Without Frontiers
   Embedding shape: (256,)
   Embedding norm: 0.883698

   Most similar plays (excluding self):
      1. [0.8837] Pillowprince - R the St

In [ ]:
# Initialize search service
search_service = FAISSSearchService(
    embeddings_path=embeddings_path,
    metadata_path=metadata_path,
    pca_path=pca_path,
    index_path=index_path,
    nlist=1024,  # Number of clusters for IVF
    nprobe=10    # Number of clusters to probe during search
)

# Initialize all components
search_service.initialize(build_index=True)


Initializing FAISS Search Service

Metadata:
  Model: sentence-transformers/multi-qa-mpnet-base-dot-v1
  Original dimension: 768
Loading PCA transformer from /Users/pooks/Dev/crate/data/pca_transformer_256d.joblib...
✓ PCA loaded: 256 components
Loading embedding model: sentence-transformers/multi-qa-mpnet-base-dot-v1...


/Users/pooks/Dev/crate/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✓ Model loaded: 768 dimensions
Loading embeddings from /Users/pooks/Dev/crate/data/embeddings_256d.npy...
✓ Loaded embeddings: (2193235, 256)
Loading existing FAISS index from /Users/pooks/Dev/crate/data/embeddings_256d.index...
✓ Index loaded: 2,193,235 vectors

✓ Initialization complete


## 2. Load Play Metadata

Load play metadata to map search results back to actual plays.


In [ ]:
# CRITICAL: Load plays in the SAME ORDER as enriched_plays_full.csv
# The embeddings were generated from the CSV in row order, so we MUST match that order

# Path to save/load alignment file (ordered play IDs)
data_dir = Path.cwd().parent.parent / "data"
alignment_file = data_dir / "play_ids_alignment.npy"
csv_path = data_dir / "enriched_plays_full.csv"  # CSV is now in data directory

# Try to load alignment file first (faster than reading CSV)
if alignment_file.exists():
    print(f"Loading play order from alignment file: {alignment_file.name}...")
    play_ids_ordered = np.load(alignment_file).tolist()
    print(f"✓ Loaded {len(play_ids_ordered):,} play IDs from alignment file")
elif csv_path.exists():
    print(f"Loading play order from {csv_path.name}...")
    # Load just the IDs to get the exact order
    csv_df = pd.read_csv(csv_path, usecols=['id'], dtype={'id': int})
    print(f"✓ CSV has {len(csv_df):,} plays")
    
    # Create ordered list of IDs
    play_ids_ordered = csv_df['id'].tolist()
    
    # Save alignment file for future use
    print(f"\nSaving alignment file to {alignment_file}...")
    np.save(alignment_file, np.array(play_ids_ordered, dtype=np.int64))
    print(f"✓ Saved alignment file ({len(play_ids_ordered):,} play IDs)")
    
    # Load full play data from database
    db = Database()
    print(f"Connected to: {db.db_path}")
    
    # Load all plays that match the CSV IDs
    # Use a chunked approach if the list is very large
    # SQLite has a limit of ~999 variables per query, so we use 500 to be safe
    print("Loading plays from database...")
    if len(play_ids_ordered) > 500:
        # For large datasets, load in chunks
        chunk_size = 500  # SQLite limit is ~999 variables, using 500 for safety
        plays_chunks = []
        for i in range(0, len(play_ids_ordered), chunk_size):
            chunk_ids = play_ids_ordered[i:i+chunk_size]
            placeholders = ','.join(['?'] * len(chunk_ids))
            chunk = db.query(f"""
                SELECT 
                    id, artist, song, album, airdate, 
                    labels, rotation_status, is_local, 
                    is_live, is_request, comment, show
                FROM fact_plays
                WHERE id IN ({placeholders})
            """, tuple(chunk_ids))
            plays_chunks.append(chunk)
        plays_df = pd.concat(plays_chunks, ignore_index=True)
    else:
        placeholders = ','.join(['?'] * len(play_ids_ordered))
        plays_df = db.query(f"""
            SELECT 
                id, artist, song, album, airdate, 
                labels, rotation_status, is_local, 
                is_live, is_request, comment, show
            FROM fact_plays
            WHERE id IN ({placeholders})
        """, tuple(play_ids_ordered))
    
    # CRITICAL: Reorder to match CSV order exactly
    id_to_order = {play_id: idx for idx, play_id in enumerate(play_ids_ordered)}
    plays_df['_csv_order'] = plays_df['id'].map(id_to_order)
    plays_df = plays_df.sort_values('_csv_order').reset_index(drop=True)
    plays_df = plays_df.drop('_csv_order', axis=1)
    
    print(f"✓ Loaded {len(plays_df):,} plays in CSV order")
else:
    print(f"⚠️  CSV not found at {csv_path}")
    print("   Loading plays with ORDER BY id (may not match embedding order!)")
    db = Database()
    print(f"Connected to: {db.db_path}")
    
    plays_df = db.query("""
        SELECT 
            id, artist, song, album, airdate, 
            labels, rotation_status, is_local, 
            is_live, is_request, comment, show
        FROM fact_plays
        WHERE artist IS NOT NULL AND song IS NOT NULL
        ORDER BY id
    """)
    print(f"✓ Loaded {len(plays_df):,} plays (ORDER BY id)")

# Verify alignment
print(f"\nAlignment check:")
print(f"   Embeddings: {len(search_service.embeddings):,}")
print(f"   Plays: {len(plays_df):,}")
if len(search_service.embeddings) == len(plays_df):
    print("   ✓ Counts match - embeddings and plays should be aligned")
else:
    print("   ⚠️  MISMATCH - truncating to match")
    min_len = min(len(search_service.embeddings), len(plays_df))
    plays_df = plays_df.iloc[:min_len].reset_index(drop=True)
    search_service.embeddings = search_service.embeddings[:min_len]
    print(f"   ✓ Aligned to {min_len:,} records")

print(f"\nSample plays (first 5):")
display(plays_df.head())


Loading play order from enriched_plays_full.csv...
✓ CSV has 2,193,235 plays

Saving alignment file to /Users/pooks/Dev/crate/data/play_ids_alignment.npy...
✓ Saved alignment file (2,193,235 play IDs)
Connected to: /Users/pooks/Dev/crate/data/music_kb.sqlite
Loading plays from database...
✓ Loaded 2,193,235 plays in CSV order

Alignment check:
   Embeddings: 2,193,235
   Plays: 2,193,235
   ✓ Counts match - embeddings and plays should be aligned

Sample plays (first 5):


,id,artist,song,album,airdate,labels,rotation_status,is_local,is_live,is_request,comment,show
0,3518527,Ami Taf Ra feat. Kamasi Washington,How I Became a Madman,The Prophet and the Madman,2025-06-25T01:49:16-07:00,"[""Brainfeeder""]",Medium,0,0,0,"North African, LA-based singer-songwriter Ami Taf Ra has announced her debut album, The Prophet and The Madman, alongside the release of new single, ""How I Became A Madman"", featuring Kamasi Washi...",63830
1,3518526,Marvin Gaye & Tammi Terrell,Ain’t No Mountain High Enough,Hitsville USA: The Motown Singles Collection 1959–1971,2025-06-25T01:46:50-07:00,"[""Motown""]",None,0,0,0,"The extraordinary Tammi Terrell died just before her 25th birthday of brain cancer. Although Marvin Gaye's relationship with her was platonic, it's said that he never got over her death.: https:/...",63830
2,3518525,Mavis Staples & Bonnie Raitt,Turn Me Around,I'll Take You There: An All-Star Concert Celebration,2025-06-25T01:42:30-07:00,"[""Blackbird Presents""]",Library,0,0,0,From Mavis Staples: I’ll Take You There — An All-Star Concert Celebration. Chicago’s Auditorium Theater 2014 \n https://www.youtube.com/watch?v=A3r63pFUk-A,63830
3,3518524,Durand Jones & The Indications,Lovers’ Holiday,Flowers,2025-06-25T01:39:11-07:00,"[""Dead Oceans""]",Light,0,0,0,October 28th at Showbox SoDo\nhttps://www.youtube.com/watch?v=Isb-OF1Iffg,63830
4,3518522,HAIM,Down to be wrong,I quit,2025-06-25T01:33:03-07:00,"[""Columbia""]",Medium,0,0,0,"HAIM will be on tour in support of their fourth album ""I quit"":\n\n-Amphitheater at McMenamin's Edgefield in Troutdale, OR on September 17th\n-WaMu Theater in Seattle on September 18th \n\nhttps:/...",63830


## 3. Test Semantic Search

Test the search service with various queries.


In [54]:
def display_search_results(query: str, indices: np.ndarray, distances: np.ndarray, top_k: int = 10):
    """Display search results in a nice format."""
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}\n")
    
    for i, (idx, dist) in enumerate(zip(indices[:top_k], distances[:top_k]), 1):
        play = plays_df.iloc[idx]
        print(f"{i:2d}. [{dist:.4f}] {play['artist']} - {play['song']}")
        if pd.notna(play.get('album')) and play['album']:
            print(f"     Album: {play['album']}")
        if pd.notna(play.get('labels')) and play['labels']:
            print(f"     Label: {play['labels']}")
        if pd.notna(play.get('rotation_status')):
            print(f"     Rotation: {play['rotation_status']}")
        if play.get('is_local') == 1:
            print(f"     🏠 Local artist")
        if pd.notna(play.get('comment')) and play['comment']:
            comment = str(play['comment'])[:100]
            print(f"     💬 {comment}..." if len(str(play['comment'])) > 100 else f"     💬 {play['comment']}")
        print()


In [61]:
# Test queries
test_queries = [
    "comic book villain london rapper",
]

for query in test_queries[:3]:  # Test first 3 queries
    try:
        indices, distances = search_service.search(query, k=10)
        display_search_results(query, indices, distances)
    except Exception as e:
        print(f"\nError searching for '{query}': {e}")
        import traceback
        traceback.print_exc()



Query: 'comic book villain london rapper'

 1. [0.5127] Madvillain - The Illest Villains
     Album: Madvillainy
     Label: ["[PIAS] UK"]
     Rotation: Library
     💬 #194.

 2. [0.4768] Viktor Vaughn - Vaudeville Villain
     Album: Vaudeville Villain
     Label: ["Sound-Ink Records"]
     💬 Alias used by DOOM based on Doctor Doom’s real name: Victor Von Doom 

https://www.discogs.com/artis...

 3. [0.4650] Theophilus London - Girls
     Album: Vibes
     Label: ["WB"]
     Rotation: Library
     💬 Theophilus Musa London is a Trinidadian-born American rapper and singer from Brooklyn, New York City...

 4. [0.4456] Madvillain - All Caps
     Album: Madvillainy
     Label: ["[PIAS] UK"]
     Rotation: Library
     💬 Madvillain is made up of emcees and producers MF Doom & Madlib.

 5. [0.4453] Madvillain - Meat Grinder
     Album: Madvillainy
     Label: ["Stones Throw Records"]
     💬 Daniel Dumile, best known by his stage name MF Doom or simply Doom, was a British-American rapper an

## 4. Search Performance Analysis

Measure search speed and analyze results.


In [56]:
import time

# Benchmark search speed
test_query = "indie rock"
n_runs = 100

print(f"Benchmarking search speed ({n_runs} queries)...")
start = time.time()

for _ in range(n_runs):
    indices, distances = search_service.search(test_query, k=10)

elapsed = time.time() - start
avg_time = elapsed / n_runs
queries_per_sec = 1 / avg_time

print(f"\nResults:")
print(f"  Total time: {elapsed:.3f}s")
print(f"  Average per query: {avg_time*1000:.2f}ms")
print(f"  Queries per second: {queries_per_sec:.1f}")


Benchmarking search speed (100 queries)...

Results:
  Total time: 1.762s
  Average per query: 17.62ms
  Queries per second: 56.8


## 5. Find Similar Plays

Find plays similar to a given play using its embedding.


In [57]:
# Pick a random play
sample_idx = 1000
sample_play = plays_df.iloc[sample_idx]

print(f"Finding plays similar to:")
print(f"  {sample_play['artist']} - {sample_play['song']}")
if pd.notna(sample_play.get('album')):
    print(f"  Album: {sample_play['album']}")
print(f"  Play ID: {sample_play['id']}\n")

# Get the embedding for this play
play_embedding = search_service.embeddings[sample_idx]

# Reshape for FAISS search
query_vector = play_embedding.reshape(1, -1)

# Search (excluding the query play itself)
distances, indices = search_service.index.search(query_vector, 11)

# Filter out the query play itself
similar_indices = [idx for idx in indices[0] if idx != sample_idx][:10]
similar_distances = [dist for idx, dist in zip(indices[0], distances[0]) if idx != sample_idx][:10]

print(f"\nTop 10 similar plays:")
for i, (idx, dist) in enumerate(zip(similar_indices, similar_distances), 1):
    play = plays_df.iloc[idx]
    print(f"{i:2d}. [{dist:.4f}] {play['artist']} - {play['song']}")


Finding plays similar to:
  The Dogs - Rock'n'Roll Part III
  Album: TOTAL DOG SHIT
  Play ID: 3517156


Top 10 similar plays:
 1. [0.3996] The Dogs - 19
 2. [0.3956] The Dogs - 19
 3. [0.3820] Slaughter and The Dogs - The Bitch
 4. [0.3709] The Dogs - John Rock & Roll Sinclair
 5. [0.3666] Slaughter and The Dogs - Situations
 6. [0.3646] that dog. - Silently
 7. [0.3611] Slaughter and The Dogs - Cranked Up Really High
 8. [0.3582] Slaughter and the Dogs - Victims of the Vampire
 9. [0.3567] Dogs - Untitled
10. [0.3562] Slaughter and the Dogs - Victims of the Vampire


## 6. Interactive Search

Try your own queries!


In [58]:
# Custom search function
def search_music(query: str, top_k: int = 10):
    """Search for music matching the query."""
    try:
        indices, distances = search_service.search(query, k=top_k)
        display_search_results(query, indices, distances, top_k=top_k)
        return indices, distances
    except Exception as e:
        print(f"Error: {e}")
        return None, None

# Example usage - try your own queries!
search_music("energetic punk rock", top_k=10)



Query: 'energetic punk rock'

 1. [0.4461] IDLES - Mr. Motivator
     Album: Live At KEXP
     Label: []
     💬 #Kev50 Highlight

Check out this smashing Live at Home session we recorded of IDLES!  https://tinyur...

 2. [0.4365] IDLES - Mr. Motivator
     Label: []
     💬 Live on KEXP at Home!

Thanks to listeners like you, KEXP has been able to host both up-and-coming, ...

 3. [0.4339] IDLES - Television
     Album: Joy as an Act of Resistance.
     Label: ["Partisan Records"]
     💬 IDLES coming to you from Kevin's basement. Coming tomorrow? Our Virtual Dance Party to some more IDL...

 4. [0.4301] Really Red - Modern Needs
     Album: Faster & Louder: Hardcore Punk, Volume 1
     Label: []
     Rotation: Library

 5. [0.4294] Culture Abuse - Dip
     Album: Dip
     Label: ["Epitaph"]
     Rotation: Library
     💬 The band describes themselves as "kinda grunge, kinda punk, kinda hardcore, definitely a good time. ...

 6. [0.4204] Accelerators - It's Cool to Rock
     Album: Singl

(array([ 119583,  549310,  580450, 1384917,  815332,  726969,  480957,
        1963903, 1283200,  642359]),
 array([0.44609678, 0.4365157 , 0.43389308, 0.43011165, 0.42935503,
        0.42042294, 0.41966885, 0.4183589 , 0.4170323 , 0.41264278],
       dtype=float32))